# Pesten — kaartspel (Flask)

Een webversie van **Pesten**, het bekende Nederlandse kaartspel. Je speelt met een gewoon
kaartspel van 52 kaarten plus 2 jokers. Wie het eerst al zijn kaarten kwijt is, wint de ronde.

**Spelverloop**
1. Iedere speler krijgt 7 kaarten (instelbaar). De bovenste kaart van de stapel gaat open.
2. Om de beurt leg je een kaart op de open stapel: **dezelfde kleur** (♥ ♦ ♣ ♠) of **dezelfde
   waarde** (7 op 7, boer op boer).
3. Kun of wil je niet, dan pak je een kaart. Past die, dan mag je hem meteen spelen; anders
   ben je klaar.
4. Heb je na je zet nog één kaart, dan meld je **"Laatste kaart!"**. Vergeet je dat, dan pak je
   2 strafkaarten.

**Pestkaarten** (elke regel kun je aan- of uitzetten)
| Kaart | Wat gebeurt er |
| --- | --- |
| 2 | De volgende speler pakt 2 kaarten |
| Joker | De volgende speler pakt 5 kaarten. Een joker mag op alles, en op een joker mag alles |
| 7 | Je bent nog een keer aan de beurt |
| 8 | De volgende speler slaat een beurt over |
| Aas | De speelrichting draait om (met twee spelers: nog een keer) |
| Boer | Je kiest de kleur waarmee verder wordt gespeeld; de boer mag op elke kaart |

Met **stapelen** mag je op een 2 of joker een 2 of joker leggen: de volgende speler pakt dan alles
bij elkaar. En je mag **niet uitgaan op een pestkaart**: je laatste kaart moet een gewone kaart zijn.

**Schermen**
1. **Aanmelden en instellingen**: e-mailadres, naam, spelvorm en tegenstanders (1 tot 5
   computers, niveau **makkelijk**, **normaal** of **moeilijk**). Online vul je de **spelcode**
   in van iemand anders, of laat je hem leeg om zelf een spel te maken. Achter het tandwiel (⚙)
   staan het aantal startkaarten, de spelregels en de assistent. Alles wordt bij je
   e-mailadres in de database bewaard (`database/pesten.db`, SQLite), samen met hoeveel
   rondes je hebt gespeeld en gewonnen.
2. **De wachtkamer** (alleen online): de spelcode en een link om te delen, wie er al meedoen.
   Wie het spel maakte, kiest hoeveel computers er meespelen en deelt de kaarten.
3. **Het spel**: jij onderaan, de anderen bovenaan, de stapels in het midden.
   Het tandwiel (⚙) rechtsboven: assistent aan/uit, nieuw spel, afmelden.

**Assistent** — geeft tijdens je beurt een tip: welke kaart je het best kunt leggen en waarom
(de kaart wordt gemarkeerd). Tegen de computer alleen op niveau makkelijk; online kan iedere
speler hem voor zichzelf aanzetten (voor beginners).

**Spelvormen**
- **Alleen tegen de computer**.
- **Online**: iedereen op een eigen telefoon, tablet of computer, met een spelcode. Er kunnen
  computers meespelen. Het hele spel staat op de server; elk apparaat krijgt alleen de eigen
  kaarten te zien. Doorgeven op één apparaat zit er bewust niet in: dan zie je elkaars kaarten.
- **Beeld en geluid** (online): met de knop *Camera aan* zien en horen de spelers elkaar.
  Dat gaat rechtstreeks tussen de apparaten (WebRTC) en werkt alleen via https.

**Bestanden**
| Bestand | Inhoud |
| --- | --- |
| `pesten_spel.ipynb` | Dit notebook: alle code van het spel |
| `start_pesten.py` | Start het spel buiten Jupyter (Docker); voert de codecellen van dit notebook uit |
| `database/pesten.db` | De database met spelers, instellingen en scores (wordt vanzelf gemaakt) |
| `templates/basis.html`, `aanmelden.html`, `wachtkamer.html`, `spel.html` | De webpagina's |
| `static/beeld.js` | Beeld en geluid tussen de spelers (WebRTC) |
| `static/kleuren.css` | Alle kleuren |
| `static/spel.css` | De opmaak |

Cellen die met `# %%ALLEEN-NOTEBOOK%%` beginnen, worden in Docker overgeslagen (installeren en testen).

## 1 — Installeren

In [ ]:
# %%ALLEEN-NOTEBOOK%% — wordt in Docker overgeslagen (daar regelt requirements.txt dit)
%pip install -q flask==3.1.3

## 2 — Configuratie
Hier pas je de spelregels aan: welke pestkaarten er zijn, hoeveel kaarten je pakt, de
computerspelers en de grenzen van de instellingen. `STANDAARD_INSTELLINGEN` is wat een nieuwe
speler op het eerste scherm ziet staan.

In [ ]:
# ==============================================================
# IMPORTS
# ==============================================================
import json
import os
import random
import re
import secrets
import signal
import sqlite3
import sys
import threading
import time
import uuid
from collections import Counter
from contextlib import contextmanager
from datetime import datetime
from pathlib import Path

from flask import Flask, flash, redirect, render_template, request, session, url_for
from werkzeug.middleware.proxy_fix import ProxyFix

# ==============================================================
# MAPPEN EN SERVER
# ==============================================================
# In Jupyter is de map van het notebook de werkmap. In Docker geeft
# start_pesten.py __file__ mee, zodat het daar ook klopt.
SPEL_MAP = Path(__file__).parent if "__file__" in globals() else Path.cwd()

# De database (SQLite) met spelers, instellingen en scores.
# Wordt vanzelf gemaakt. In Docker staat hij in een eigen volume.
DATABASE = Path(os.getenv("SPEL_DB", SPEL_MAP / "database" / "pesten.db"))

SERVER_HOST = os.getenv("SPEL_HOST", "127.0.0.1")
SERVER_PORT = int(os.getenv("SPEL_PORT", "5000"))
# Achter Caddy (https) zet docker-compose.yml SPEL_HTTPS=1: de browser stuurt het
# sessie-cookie dan alleen over https. In Jupyter (http) staat dit uit.
SPEL_HTTPS = os.getenv("SPEL_HTTPS") == "1"

# Wie mag spelen: e-mailadressen en/of hele domeinen (met @ ervoor), gescheiden door
# komma's, bv. "anna@gmail.com, bas@hotmail.com, @familie.nl". Komt uit .env (zie
# docker-compose.yml). Leeg = iedereen mag spelen (handig in Jupyter).
TOEGESTANE_EMAILS = {deel.strip().lower() for deel in os.getenv("TOEGESTANE_EMAILS", "").split(",")
                     if deel.strip()}

# Auditlog: aanmeldingen en mislukte pogingen in de database (zie sectie 4).
AUDIT_BEWAAR_DAGEN = 90

# ==============================================================
# SPELREGELS
# ==============================================================
# Elke regel kan de speler op het eerste scherm aan- of uitzetten.
# sleutel: tekst op het scherm
REGELS = {
    "twee": "2 — de volgende speler pakt 2 kaarten",
    "joker": "Joker — de volgende speler pakt 5 kaarten (zonder deze regel: geen jokers in het spel)",
    "zeven": "7 — je bent nog een keer aan de beurt",
    "acht": "8 — de volgende speler slaat een beurt over",
    "aas": "Aas — de speelrichting draait om",
    "boer": "Boer — je kiest de kleur waarmee verder wordt gespeeld",
    "boer_op_alles": "Een boer mag op elke kaart",
    "stapelen": "Stapelen: op een 2 of joker mag een 2 of joker (de volgende pakt alles)",
    "niet_uitgaan_pest": "Je mag niet uitgaan op een pestkaart",
    "laatste_kaart": "\"Laatste kaart!\" melden, anders 2 strafkaarten",
}
TWEE_PAKKEN = 2          # kaarten die je pakt door een 2
JOKER_PAKKEN = 5         # kaarten die je pakt door een joker
STRAF_LAATSTE_KAART = 2  # strafkaarten als je "Laatste kaart!" vergeet

SPELVORMEN = {
    "alleen": "Alleen tegen de computer",
    "online": "Online — iedereen speelt op een eigen telefoon, tablet of computer",
}

# Computerspelers: hoe slim ze spelen en hoe vaak ze "Laatste kaart!" vergeten.
NIVEAUS = {
    "makkelijk": "Makkelijk — legt zomaar een kaart die past",
    "normaal": "Normaal — bewaart jokers en boeren voor later",
    "moeilijk": "Moeilijk — pest gericht als iemand bijna uit is",
}
# De assistent (tips tijdens het spel) is er tegen de computer alleen op deze
# niveaus. Online kan elke speler hem voor zichzelf aanzetten (voor beginners).
ASSISTENT_NIVEAUS = {"makkelijk"}
VERGEET_KANS = {"makkelijk": 0.4, "normaal": 0.15, "moeilijk": 0.0}
COMPUTER_NAMEN = ["Anna", "Bram", "Cor", "Dewi", "Emre"]
COMPUTER_PAUZE_MS = 900   # zo lang wacht de pagina tussen twee zetten van de computer

MIN_COMPUTERS, MAX_COMPUTERS = 1, len(COMPUTER_NAMEN)   # online mag het ook 0 zijn
MAX_SPELERS = 6           # mensen en computers samen aan één tafel
MIN_STARTKAARTEN, MAX_STARTKAARTEN = 5, 10
OVER_IN_STAPEL = 10       # na het delen moeten er minstens zoveel kaarten in de stapel blijven
MAX_NAAM = 20             # maximaal aantal tekens van je naam

# Online spelen
CODE_TEKENS = "ABCDEFGHJKLMNPQRSTUVWXYZ23456789"   # zonder 0/O en 1/I: die lijken op elkaar
CODE_LENGTE = 5           # zo lang is de spelcode die je aan de anderen geeft
VERVERS_MS = 1500         # zo vaak kijkt de pagina of een ander een zet heeft gedaan
BEELD_VERVERS_MS = 1000   # beeld en geluid: zo vaak haalt de pagina berichten op
BEELD_WEG_NA = 15         # seconden: wie zo lang niets ophaalt, is uit het gesprek

STANDAARD_INSTELLINGEN = {
    "spelvorm": "alleen",
    "assistent": True,        # tips tijdens het spel (zie sectie 6)
    "computers": 3,
    "niveau": "normaal",
    "startkaarten": 7,
    "regels": {regel: True for regel in REGELS},
}

print(f"[Config] database      : {DATABASE}")
print(f"[Config] toegang       : {', '.join(sorted(TOEGESTANE_EMAILS)) or 'iedereen'}")
print(f"[Config] regels        : {', '.join(REGELS)}")
print(f"[Config] tegenstanders : {MIN_COMPUTERS}-{MAX_COMPUTERS}, startkaarten {MIN_STARTKAARTEN}-{MAX_STARTKAARTEN}")

## 3 — Kaarten
Een kaart is een korte tekst: de kleur (`H` harten, `S` schoppen, `R` ruiten, `K` klaveren)
en dan de waarde (`2`…`10`, `B` boer, `V` vrouw, `H` heer, `A` aas). `X1` en `X2` zijn de jokers.
Of een kaart een pestkaart is, hangt af van de regels die aan staan (`pestsoort`).

In [ ]:
# ==============================================================
# KAARTEN
# ==============================================================
# Een kaart is een korte tekst: eerst de kleur, dan de waarde.
#   "H7" = harten 7,  "S10" = schoppen 10,  "KB" = klaveren boer,  "RA" = ruiten aas
#   "X1" en "X2" = de twee jokers
# In de volgorde hieronder (rood, zwart, rood, zwart) wordt ook je hand gesorteerd.
KLEUREN = {"H": "harten", "S": "schoppen", "R": "ruiten", "K": "klaveren"}
SYMBOLEN = {"H": "♥", "S": "♠", "R": "♦", "K": "♣"}
RODE_KLEUREN = {"H", "R"}
WAARDEN = ["2", "3", "4", "5", "6", "7", "8", "9", "10", "B", "V", "H", "A"]
WAARDE_NAMEN = {"B": "boer", "V": "vrouw", "H": "heer", "A": "aas"}
JOKER = "X"

# Welke waarde welke pestregel heeft (zie REGELS in sectie 2).
PESTWAARDEN = {"2": "twee", "7": "zeven", "8": "acht", "A": "aas", "B": "boer"}


def nieuw_pak(met_jokers: bool) -> list[str]:
    """Een geschud pak: 52 kaarten, plus 2 jokers als die meedoen."""
    pak = [kleur + waarde for kleur in KLEUREN for waarde in WAARDEN]
    if met_jokers:
        pak += [JOKER + "1", JOKER + "2"]
    random.shuffle(pak)
    return pak


def kleur_van(kaart: str) -> str:
    return kaart[0]


def waarde_van(kaart: str) -> str:
    return kaart[1:]


def is_joker(kaart: str) -> bool:
    return kaart[0] == JOKER


def kaart_naam(kaart: str) -> str:
    """'harten 7', 'schoppen boer' of 'joker'."""
    if is_joker(kaart):
        return "joker"
    waarde = waarde_van(kaart)
    return f"{KLEUREN[kleur_van(kaart)]} {WAARDE_NAMEN.get(waarde, waarde)}"


def pestsoort(kaart: str, regels: dict) -> str | None:
    """Welke pestregel deze kaart heeft ('twee', 'joker', ...), of None voor een
    gewone kaart. Een regel die uit staat, maakt de kaart gewoon."""
    soort = "joker" if is_joker(kaart) else PESTWAARDEN.get(waarde_van(kaart))
    return soort if soort and regels.get(soort) else None


def sorteer_hand(hand: list[str]) -> list[str]:
    """Op kleur (♥ ♠ ♦ ♣) en daarbinnen op waarde; jokers achteraan."""
    volgorde = list(KLEUREN)
    return sorted(hand, key=lambda k: (len(volgorde), k) if is_joker(k)
                  else (volgorde.index(kleur_van(k)), WAARDEN.index(waarde_van(k))))

In [ ]:
# %%ALLEEN-NOTEBOOK%% — testcel: een pak kaarten
pak = nieuw_pak(met_jokers=True)
print(f"{len(pak)} kaarten, bijvoorbeeld: {', '.join(kaart_naam(k) for k in pak[:5])}")
print("Gesorteerd:", sorteer_hand(pak[:10]))
print("Pestsoorten:", {k: pestsoort(k, STANDAARD_INSTELLINGEN["regels"]) for k in ["H2", "X1", "S7", "KB", "R5"]})

## 4 — Database
Eén tabel `spelers` in een SQLite-bestand. Per e-mailadres:
- de **naam** die de andere spelers zien,
- de **instellingen** van het eerste scherm (als JSON),
- hoeveel rondes de speler heeft **gespeeld** en **gewonnen**.

Komt er later een regel bij, dan vult `aangevulde_instellingen` die aan met de standaard.

`email_toegestaan` kijkt of een e-mailadres op de lijst `TOEGESTANE_EMAILS` staat (sectie 2).

Een tweede tabel `audit` is het **auditlog**: elke aanmelding en elke mislukte poging, met tijd,
e-mailadres, IP-adres en browser. Zo zie je of iemand probeert binnen te komen die niet op de
lijst staat. Regels ouder dan `AUDIT_BEWAAR_DAGEN` worden opgeruimd. Bekijken: `./audit.sh`
(haalt het log van de Pi), of in Jupyter met `audit_regels()`.

| Soort | Wanneer |
| --- | --- |
| `aangemeld` | Iemand meldt zich aan en start of doet mee met een spel |
| `afgemeld` | Iemand meldt zich af |
| `geweigerd` | Aanmelden met een adres dat niet op de lijst staat |
| `opzoeken_geweigerd` | Een adres dat niet op de lijst staat, wordt ingevuld (nog niet op Start geklikt) |
| `ongeldig_adres` | Aanmelden met iets wat geen e-mailadres is |
| `onbekende_code` | Meedoen met een spelcode die niet bestaat (raden van codes?) |
| `toegang_ingetrokken` | Iemand die van de lijst is gehaald, wordt afgemeld |

In [ ]:
# ==============================================================
# DATABASE (SQLite)
# ==============================================================
EMAIL_PATROON = re.compile(r"[^@\s]+@[^@\s]+\.[^@\s]+")


@contextmanager
def database():
    """Verbinding met de database: opslaan als alles goed gaat, terugdraaien bij een fout."""
    DATABASE.parent.mkdir(parents=True, exist_ok=True)
    verbinding = sqlite3.connect(DATABASE, timeout=10)
    verbinding.row_factory = sqlite3.Row
    try:
        with verbinding:
            yield verbinding
    finally:
        verbinding.close()


def maak_database() -> None:
    with database() as db:
        db.execute(f"""
            CREATE TABLE IF NOT EXISTS spelers (
                email        TEXT PRIMARY KEY,
                naam         TEXT NOT NULL DEFAULT '' CHECK (length(naam) <= {MAX_NAAM}),
                instellingen TEXT NOT NULL DEFAULT '{{}}',
                gespeeld     INTEGER NOT NULL DEFAULT 0,
                gewonnen     INTEGER NOT NULL DEFAULT 0,
                aangemaakt   TEXT NOT NULL,
                bijgewerkt   TEXT NOT NULL
            )""")
        db.execute("""
            CREATE TABLE IF NOT EXISTS audit (
                id      INTEGER PRIMARY KEY,
                tijd    TEXT NOT NULL,
                soort   TEXT NOT NULL,
                email   TEXT NOT NULL DEFAULT '',
                ip      TEXT NOT NULL DEFAULT '',
                details TEXT NOT NULL DEFAULT '',
                browser TEXT NOT NULL DEFAULT ''
            )""")
        db.execute("CREATE INDEX IF NOT EXISTS audit_tijd ON audit (tijd)")


def nu() -> str:
    return datetime.now().isoformat(sep=" ", timespec="seconds")


def lees_email(tekst: str) -> str:
    """Controleert het e-mailadres en geeft het in kleine letters terug."""
    email = tekst.strip().lower()
    if not email:
        raise ValueError("Vul je e-mailadres in.")
    if len(email) > 254 or not EMAIL_PATROON.fullmatch(email):
        raise ValueError(f"'{email[:60]}' is geen geldig e-mailadres.")
    return email


def email_toegestaan(email: str) -> bool:
    """Staat dit adres (of zijn domein) op TOEGESTANE_EMAILS? Een lege lijst laat iedereen toe."""
    if not TOEGESTANE_EMAILS:
        return True
    return email in TOEGESTANE_EMAILS or "@" + email.rsplit("@", 1)[-1] in TOEGESTANE_EMAILS


def lees_naam(tekst: str) -> str:
    naam = " ".join(tekst.split())
    if not naam:
        raise ValueError("Vul je naam in; die zien de andere spelers.")
    if len(naam) > MAX_NAAM:
        raise ValueError(f"Je naam mag maximaal {MAX_NAAM} tekens zijn.")
    return naam


def bewaar_speler(email: str, naam: str, instellingen: dict) -> None:
    """Voegt een speler toe of werkt naam en instellingen bij."""
    with database() as db:
        db.execute("""
            INSERT INTO spelers (email, naam, instellingen, aangemaakt, bijgewerkt)
            VALUES (?, ?, ?, ?, ?)
            ON CONFLICT (email) DO UPDATE SET
                naam = excluded.naam,
                instellingen = excluded.instellingen,
                bijgewerkt = excluded.bijgewerkt""",
            (email, naam, json.dumps(instellingen), nu(), nu()))


def haal_speler(email: str) -> dict | None:
    """De speler met zijn instellingen (aangevuld met de standaard), of None."""
    with database() as db:
        rij = db.execute("SELECT * FROM spelers WHERE email = ?", (email,)).fetchone()
    if not rij:
        return None
    speler = dict(rij)
    try:
        opgeslagen = json.loads(speler["instellingen"])
    except json.JSONDecodeError:
        opgeslagen = {}
    speler["instellingen"] = aangevulde_instellingen(opgeslagen)
    return speler


def aangevulde_instellingen(opgeslagen: dict) -> dict:
    """Opgeslagen instellingen, aangevuld met de standaard (bv. als er een regel bij komt)."""
    instellingen = {**STANDAARD_INSTELLINGEN, **opgeslagen}
    instellingen["regels"] = {**STANDAARD_INSTELLINGEN["regels"], **opgeslagen.get("regels", {})}
    return instellingen


def tel_ronde(email: str, gewonnen: bool) -> None:
    """Telt een gespeelde ronde (en een gewonnen ronde) bij de speler op."""
    with database() as db:
        db.execute("UPDATE spelers SET gespeeld = gespeeld + 1, gewonnen = gewonnen + ?, "
                   "bijgewerkt = ? WHERE email = ?", (int(gewonnen), nu(), email))


# Mislukte pogingen; de rest van het auditlog is ter informatie.
AUDIT_FOUTEN = ("geweigerd", "opzoeken_geweigerd", "ongeldig_adres", "onbekende_code", "toegang_ingetrokken")


def audit(soort: str, email: str = "", ip: str = "", details: str = "", browser: str = "") -> None:
    """Schrijft een regel in het auditlog (en in het logboek van de server / docker logs)."""
    tijd = nu()
    with database() as db:
        db.execute("INSERT INTO audit (tijd, soort, email, ip, details, browser) VALUES (?, ?, ?, ?, ?, ?)",
                   (tijd, soort, email[:254], ip[:64], details[:200], browser[:200]))
        if random.random() < 0.02:   # af en toe oude regels opruimen
            grens = datetime.fromtimestamp(time.time() - AUDIT_BEWAAR_DAGEN * 86400)
            db.execute("DELETE FROM audit WHERE tijd < ?", (grens.isoformat(sep=" ", timespec="seconds"),))
    print(f"[Audit] {tijd} {soort:<20} {email or '-':<30} {ip or '-':<15} {details}", flush=True)


def audit_regels(dagen: int = 7, alleen_fouten: bool = True) -> list[dict]:
    """De regels van het auditlog van de laatste dagen, nieuwste eerst."""
    grens = datetime.fromtimestamp(time.time() - dagen * 86400).isoformat(sep=" ", timespec="seconds")
    sql = "SELECT tijd, soort, email, ip, details, browser FROM audit WHERE tijd >= ?"
    if alleen_fouten:
        sql += f" AND soort IN ({', '.join('?' * len(AUDIT_FOUTEN))})"
    with database() as db:
        rijen = db.execute(sql + " ORDER BY id DESC",
                           (grens, *(AUDIT_FOUTEN if alleen_fouten else ()))).fetchall()
    return [dict(r) for r in rijen]


maak_database()
with database() as db:
    print(f"[Database] {DATABASE}  ({db.execute('SELECT count(*) FROM spelers').fetchone()[0]} spelers)")

## 5 — Spellogica
Het spel is een gewone `dict` met de spelers (elk met een `hand`), de trekstapel, de open
stapel, wie er aan de beurt is en de richting. Drie zetten veranderen het spel:

- `speel_kaart` — leg een kaart (bij een boer met een kleur, eventueel met "Laatste kaart!")
- `pak_kaart` — pak de strafkaarten, of één kaart (die je daarna nog mag spelen)
- `pas` — de gepakte kaart toch niet spelen

`waarom_niet` zegt waarom een kaart niet mag; zo krijgt de speler een duidelijke melding.
Elke speler heeft een `soort`: `mens` of `computer`. Tegen de computer en online gebruiken
dezelfde functies; online zijn er gewoon meer mensen.

**Online** begint een spel in de **wachtkamer** (fase `wachten`): `meedoen` zet een mens aan
tafel, `kies_computers` zet er computers bij en `begin_spel` deelt de kaarten. Gaat iemand
weg (`verlaat`), dan speelt een computer verder met die kaarten.

Deze cel kent geen Flask: je kunt het spel hier los testen (zie de testcel verderop).

In [ ]:
# ==============================================================
# SPELLOGICA
# ==============================================================
class SpelFout(Exception):
    """Een zet die nu niet mag (bv. een kaart die niet past)."""


MAX_LOG = 40   # zoveel regels van het logboek worden bewaard


def controleer_instellingen(instellingen: dict) -> dict:
    """Controleert de instellingen van het eerste scherm en geeft ze netjes terug."""
    spelvorm = instellingen.get("spelvorm", "alleen")
    if spelvorm not in SPELVORMEN:
        raise ValueError(f"Onbekende spelvorm: {spelvorm}")
    try:
        computers = int(instellingen.get("computers", 0))
        startkaarten = int(instellingen.get("startkaarten", 0))
    except (TypeError, ValueError):
        raise ValueError("Het aantal tegenstanders en startkaarten moeten getallen zijn.")
    min_computers = 0 if spelvorm == "online" else MIN_COMPUTERS
    if not min_computers <= computers <= MAX_COMPUTERS:
        raise ValueError(f"Kies {min_computers} tot {MAX_COMPUTERS} computerspelers.")
    if not MIN_STARTKAARTEN <= startkaarten <= MAX_STARTKAARTEN:
        raise ValueError(f"Kies {MIN_STARTKAARTEN} tot {MAX_STARTKAARTEN} startkaarten.")
    niveau = instellingen.get("niveau")
    if niveau not in NIVEAUS:
        raise ValueError(f"Onbekend niveau: {niveau}")
    regels = {regel: bool(instellingen.get("regels", {}).get(regel)) for regel in REGELS}

    pak = 52 + (2 if regels["joker"] else 0)
    if (computers + 1) * startkaarten > pak - OVER_IN_STAPEL:
        raise ValueError(f"Te veel kaarten: {computers + 1} spelers × {startkaarten} kaarten past niet "
                         f"in een pak van {pak}. Kies minder tegenstanders of startkaarten.")
    return {"spelvorm": spelvorm, "computers": computers, "niveau": niveau,
            "startkaarten": startkaarten, "regels": regels,
            "assistent": bool(instellingen.get("assistent"))
                         and (spelvorm == "online" or niveau in ASSISTENT_NIVEAUS)}


def assistent_mogelijk(spel: dict) -> bool:
    """Online mag iedereen de assistent aanzetten; tegen de computer alleen op ASSISTENT_NIVEAUS."""
    return spel["online"] or spel["niveau"] in ASSISTENT_NIVEAUS


def nieuw_spel(naam: str, instellingen: dict, email: str | None = None) -> dict:
    """Een spel met jou op plaats 0. Alleen tegen de computer schuiven de computers
    meteen aan en worden de kaarten gedeeld. Online wacht het spel eerst in de
    wachtkamer tot de anderen meedoen (zie meedoen en begin_spel)."""
    instellingen = controleer_instellingen(instellingen)
    spel = {
        "spelers": [],
        "online": instellingen["spelvorm"] == "online",
        "computers": instellingen["computers"],   # zoveel computers schuiven aan bij begin_spel
        "regels": instellingen["regels"],
        "startkaarten": instellingen["startkaarten"],
        "niveau": instellingen["niveau"],
        "stand": [],                  # gewonnen rondes per speler
        "ronde": 0,
        "fase": "wachten",            # wachten -> spelen -> klaar
        "log": [],
        "gestart": time.time(),
    }
    meedoen(spel, naam, email, instellingen["assistent"])
    if not spel["online"]:
        begin_spel(spel, 0)
    return spel


def meedoen(spel: dict, naam: str, email: str | None, assistent: bool = False) -> int:
    """Een mens schuift aan in de wachtkamer; geeft zijn plaats aan tafel. Wie al
    meedoet (zelfde e-mailadres, bv. op een ander apparaat), krijgt die plaats terug."""
    for i, speler in enumerate(spel["spelers"]):
        if email and speler["soort"] == "mens" and speler["email"] == email:
            return i
    if spel["fase"] != "wachten":
        raise SpelFout("Dit spel is al begonnen. Meedoen kan alleen zolang het in de wachtkamer staat.")
    if len(spel["spelers"]) >= MAX_SPELERS:
        raise SpelFout(f"Dit spel is vol: er kunnen {MAX_SPELERS} spelers aan tafel.")
    namen = {s["naam"].lower() for s in spel["spelers"]}
    uniek, nummer = naam, 2
    while uniek.lower() in namen:
        uniek, nummer = f"{naam} {nummer}", nummer + 1
    spel["spelers"].append({
        "naam": uniek, "soort": "mens", "email": email,
        "id": secrets.token_hex(4),   # voor beeld en geluid; zo zien anderen je e-mailadres niet
        "assistent": bool(assistent) and assistent_mogelijk(spel),
    })
    if spel["online"]:
        meld(spel, f"{uniek} doet mee.", "info")
    return len(spel["spelers"]) - 1


def computer_namen(spel: dict, aantal: int) -> list[str]:
    """Namen voor de computers, het liefst namen die nog niemand aan tafel heeft."""
    bezet = {s["naam"].lower() for s in spel["spelers"]}
    vrij = [n for n in COMPUTER_NAMEN if n.lower() not in bezet]
    return (vrij + [n + " (computer)" for n in COMPUTER_NAMEN if n.lower() in bezet])[:aantal]


def computers_erbij(spel: dict) -> int:
    """Hoeveel computers er bij begin_spel aanschuiven (de tafel heeft MAX_SPELERS plaatsen)."""
    return min(spel["computers"], MAX_SPELERS - len(spel["spelers"]))


def kies_computers(spel: dict, idx: int, aantal: int) -> None:
    """In de wachtkamer kiest wie het spel maakte (plaats 0) hoeveel computers er meespelen."""
    if spel["fase"] != "wachten":
        raise SpelFout("Het spel is al begonnen.")
    if idx != 0:
        raise SpelFout(f"Alleen {spel['spelers'][0]['naam']} kan het aantal computers kiezen.")
    if not 0 <= aantal <= MAX_COMPUTERS:
        raise SpelFout(f"Kies 0 tot {MAX_COMPUTERS} computerspelers.")
    spel["computers"] = aantal


def begin_spel(spel: dict, idx: int) -> None:
    """De computers schuiven aan en de eerste ronde begint. Online mag alleen wie
    het spel maakte (plaats 0) dit doen."""
    if spel["fase"] != "wachten":
        raise SpelFout("Het spel is al begonnen.")
    if idx != 0:
        raise SpelFout(f"Wacht tot {spel['spelers'][0]['naam']} het spel begint.")
    aantal = computers_erbij(spel)
    if len(spel["spelers"]) + aantal < 2:
        raise SpelFout("Je hebt minstens één tegenstander nodig: wacht op een medespeler "
                       "of zet er een computer bij.")
    for computer_naam in computer_namen(spel, aantal):
        spel["spelers"].append({"naam": computer_naam, "soort": "computer", "niveau": spel["niveau"]})

    # Met veel spelers past het aantal startkaarten soms niet: dan krijgt iedereen er wat minder.
    pak_grootte = 52 + (2 if spel["regels"]["joker"] else 0)
    past = (pak_grootte - OVER_IN_STAPEL) // len(spel["spelers"])
    minder = spel["startkaarten"] > past
    spel["startkaarten"] = min(spel["startkaarten"], past)
    spel["stand"] = [0] * len(spel["spelers"])
    spel["starter"] = random.randrange(len(spel["spelers"]))
    nieuwe_ronde(spel)
    if minder:
        meld(spel, f"Met {len(spel['spelers'])} spelers krijgt iedereen {spel['startkaarten']} kaarten.", "info")


def verlaat(spel: dict, idx: int) -> bool:
    """Speler idx gaat weg. In de wachtkamer verdwijnt hij van de lijst; tijdens het
    spel speelt een computer verder met zijn kaarten. True als er geen mensen meer zijn."""
    speler = spel["spelers"][idx]
    if spel["fase"] == "wachten":
        del spel["spelers"][idx]
        meld(spel, f"{speler['naam']} is weggegaan.", "info")
    else:
        speler.update(soort="computer", niveau="normaal", email=None, assistent=False)
        meld(spel, f"{speler['naam']} is weggegaan; de computer speelt verder met die kaarten.", "info")
    return not any(s["soort"] == "mens" for s in spel["spelers"])


def nieuwe_ronde(spel: dict) -> None:
    """Schudt, deelt en draait de eerste kaart om. Elke ronde begint de volgende speler."""
    pak = nieuw_pak(met_jokers=spel["regels"]["joker"])
    for speler in spel["spelers"]:
        speler["hand"] = [pak.pop() for _ in range(spel["startkaarten"])]
        speler["nieuw"] = []   # net gepakte kaarten (lichten op in je hand)

    # De eerste open kaart is een gewone kaart: een pestkaart gaat terug in de stapel.
    while pestsoort(pak[-1], spel["regels"]):
        pak.insert(random.randrange(len(pak) - 1), pak.pop())
    eerste = pak.pop()

    if spel["ronde"]:
        spel["starter"] = (spel["starter"] + 1) % len(spel["spelers"])
    spel.update({
        "trekstapel": pak,
        "aflegstapel": [eerste],
        "kleur": kleur_van(eerste),   # de kleur die gevolgd moet worden (None na een joker)
        "beurt": spel["starter"],
        "richting": 1,                # 1 = met de klok mee, -1 = tegen de klok in
        "te_pakken": 0,               # opgestapelde 2'en en jokers
        "getrokken": None,            # kaart die de speler aan de beurt net pakte en nog mag spelen
        "extra_beurt": False,         # na een 7
        "fase": "spelen",             # spelen -> klaar
        "winnaar": None,
        "log": [],
        "ronde": spel["ronde"] + 1,
    })
    meld(spel, f"Ronde {spel['ronde']}: {kaart_naam(eerste)} ligt open. "
               f"{spel['spelers'][spel['beurt']]['naam']} begint.", "ronde")


def meld(spel: dict, tekst: str, soort: str = "zet") -> None:
    spel["log"].append({"tekst": tekst, "soort": soort})
    del spel["log"][:-MAX_LOG]


def bovenste(spel: dict) -> str:
    return spel["aflegstapel"][-1]


def aan_de_beurt(spel: dict) -> dict:
    return spel["spelers"][spel["beurt"]]


def controleer_beurt(spel: dict, idx: int) -> None:
    if spel["fase"] != "spelen":
        raise SpelFout("De ronde is afgelopen.")
    if spel["beurt"] != idx:
        raise SpelFout(f"{aan_de_beurt(spel)['naam']} is aan de beurt, niet jij.")


def waarom_niet(spel: dict, kaart: str) -> str | None:
    """Waarom de speler aan de beurt deze kaart niet mag spelen, of None als het mag."""
    regels = spel["regels"]
    hand = aan_de_beurt(spel)["hand"]
    soort = pestsoort(kaart, regels)
    if spel["getrokken"] and kaart != spel["getrokken"]:
        return "Je hebt een kaart gepakt; alleen die mag je nog spelen. Of pas."
    if regels["niet_uitgaan_pest"] and len(hand) == 1 and soort:
        return "Je mag niet uitgaan op een pestkaart. Pak een kaart."
    if spel["te_pakken"]:
        if regels["stapelen"] and soort in ("twee", "joker"):
            return None
        return (f"Je moet {spel['te_pakken']} kaarten pakken"
                + (", of een 2 of joker bijleggen." if regels["stapelen"] else "."))
    if soort == "joker" or (soort == "boer" and regels["boer_op_alles"]):
        return None
    if spel["kleur"] is None:   # op een joker mag alles
        return None
    top = bovenste(spel)
    if kleur_van(kaart) == spel["kleur"]:
        return None
    if not is_joker(top) and not is_joker(kaart) and waarde_van(kaart) == waarde_van(top):
        return None
    gevraagd = f"{SYMBOLEN[spel['kleur']]} {KLEUREN[spel['kleur']]}"
    return f"{kaart_naam(kaart).capitalize()} past niet: leg {gevraagd} of dezelfde waarde."


def speelbare_kaarten(spel: dict) -> list[str]:
    return [k for k in aan_de_beurt(spel)["hand"] if waarom_niet(spel, k) is None]


def pak(spel: dict, speler: dict, aantal: int) -> list[str]:
    """Pakt kaarten van de stapel. Is die op, dan wordt de open stapel (behalve
    de bovenste kaart) geschud en de nieuwe trekstapel."""
    gepakt = []
    for _ in range(aantal):
        if not spel["trekstapel"]:
            spel["trekstapel"] = spel["aflegstapel"][:-1]
            spel["aflegstapel"] = spel["aflegstapel"][-1:]
            random.shuffle(spel["trekstapel"])
            if spel["trekstapel"]:
                meld(spel, "De stapel is op: de open kaarten zijn geschud.", "info")
        if not spel["trekstapel"]:
            break   # alle kaarten zijn in handen van spelers
        gepakt.append(spel["trekstapel"].pop())
    speler["hand"] += gepakt
    speler["nieuw"] = gepakt
    return gepakt


def volgende(spel: dict, overslaan: int = 0) -> None:
    aantal = len(spel["spelers"])
    spel["beurt"] = (spel["beurt"] + spel["richting"] * (1 + overslaan)) % aantal
    spel["getrokken"] = None
    spel["extra_beurt"] = False


def kaarten_tekst(n: int) -> str:
    return f"{n} {'kaart' if n == 1 else 'kaarten'}"


def speel_kaart(spel: dict, idx: int, kaart: str, gekozen_kleur: str | None = None,
                laatste_gemeld: bool = False) -> None:
    """Speler idx legt een kaart op de open stapel. Bij een boer kies je een kleur;
    met laatste_gemeld meld je "Laatste kaart!" als je daarna nog één kaart hebt."""
    controleer_beurt(spel, idx)
    speler = spel["spelers"][idx]
    if kaart not in speler["hand"]:
        raise SpelFout("Die kaart heb je niet.")
    reden = waarom_niet(spel, kaart)
    if reden:
        raise SpelFout(reden)
    regels = spel["regels"]
    soort = pestsoort(kaart, regels)
    if soort == "boer" and gekozen_kleur not in KLEUREN:
        raise SpelFout("Kies een kleur bij de boer.")

    speler["hand"].remove(kaart)
    speler["nieuw"] = []
    spel["aflegstapel"].append(kaart)
    spel["getrokken"] = None
    if soort == "boer":
        spel["kleur"] = gekozen_kleur
    else:
        spel["kleur"] = None if is_joker(kaart) else kleur_van(kaart)

    tekst = f"{speler['naam']} legt {kaart_naam(kaart)}"
    if soort == "twee":
        spel["te_pakken"] += TWEE_PAKKEN
    elif soort == "joker":
        spel["te_pakken"] += JOKER_PAKKEN
    if soort in ("twee", "joker"):
        tekst += f": de volgende pakt {spel['te_pakken']}"
    elif soort == "boer":
        tekst += f" en kiest {SYMBOLEN[gekozen_kleur]} {KLEUREN[gekozen_kleur]}"
    meld(spel, tekst + ".", "pest" if soort else "zet")

    if len(speler["hand"]) == 1:
        if laatste_gemeld or not regels["laatste_kaart"]:
            meld(spel, f"{speler['naam']}: \"Laatste kaart!\"", "laatste")
        else:
            pak(spel, speler, STRAF_LAATSTE_KAART)
            meld(spel, f"{speler['naam']} vergat \"Laatste kaart!\" te melden en pakt "
                       f"{kaarten_tekst(STRAF_LAATSTE_KAART)} straf.", "straf")

    if not speler["hand"]:
        spel["fase"] = "klaar"
        spel["winnaar"] = idx
        spel["stand"][idx] += 1
        meld(spel, f"{speler['naam']} is uit en wint ronde {spel['ronde']}!", "winst")
        return

    aantal = len(spel["spelers"])
    if soort == "zeven":
        spel["extra_beurt"] = True
        meld(spel, f"{speler['naam']} is nog een keer aan de beurt.", "info")
    elif soort == "aas":
        spel["richting"] *= -1
        if aantal == 2:
            spel["extra_beurt"] = True
            meld(spel, f"De richting draait om: {speler['naam']} is nog een keer aan de beurt.", "info")
        else:
            meld(spel, "De richting draait om.", "info")
            volgende(spel)
    elif soort == "acht":
        overgeslagen = spel["spelers"][(idx + spel["richting"]) % aantal]
        meld(spel, f"{overgeslagen['naam']} slaat een beurt over.", "info")
        volgende(spel, overslaan=1)
    else:
        volgende(spel)


def pak_kaart(spel: dict, idx: int) -> None:
    """Speler idx pakt van de stapel: de opgestapelde strafkaarten, of één kaart.
    Past die ene kaart, dan mag hij hem nog spelen (of passen); anders is de beurt voorbij."""
    controleer_beurt(spel, idx)
    if spel["getrokken"]:
        raise SpelFout("Je hebt al een kaart gepakt. Speel hem of pas.")
    speler = spel["spelers"][idx]

    if spel["te_pakken"]:
        gepakt = pak(spel, speler, spel["te_pakken"])
        spel["te_pakken"] = 0
        meld(spel, f"{speler['naam']} pakt {kaarten_tekst(len(gepakt))}.", "straf")
        volgende(spel)
        return

    gepakt = pak(spel, speler, 1)
    if not gepakt:
        meld(spel, f"Er zijn geen kaarten meer om te pakken; {speler['naam']} past.", "info")
        volgende(spel)
        return
    spel["getrokken"] = gepakt[0]
    if waarom_niet(spel, gepakt[0]) is None:
        meld(spel, f"{speler['naam']} pakt een kaart.")
    else:
        meld(spel, f"{speler['naam']} pakt een kaart en past.")
        volgende(spel)


def pas(spel: dict, idx: int) -> None:
    """Na het pakken de gepakte kaart toch niet spelen."""
    controleer_beurt(spel, idx)
    if not spel["getrokken"]:
        raise SpelFout("Je kunt pas passen nadat je een kaart hebt gepakt.")
    meld(spel, f"{spel['spelers'][idx]['naam']} past.")
    volgende(spel)

## 6 — Computerspelers en assistent
Drie niveaus (zie `NIVEAUS` in sectie 2):
- **Makkelijk** legt een willekeurige kaart die past en vergeet vaak "Laatste kaart!".
- **Normaal** bewaart jokers en boeren, legt een 7 alleen als er een vervolg is, stapelt
  liever met een 2 dan met een joker en houdt kleuren waar hij veel van heeft.
- **Moeilijk** doet hetzelfde, maar pest gericht (2, 8, joker) als de volgende speler bijna
  uit is, en vergeet "Laatste kaart!" nooit.

De testcel speelt honderden spellen met alleen computers: zo zie je dat elk spel stopt,
dat er geen kaarten zoekraken en of "moeilijk" echt beter speelt.

**Assistent** — tegen de computer alleen op niveau **makkelijk** (`ASSISTENT_NIVEAUS`); online kan
elke speler hem voor zichzelf aanzetten (handig voor beginners, alleen jij ziet je tips).
Staat hij aan (tandwiel op het eerste scherm of in het spel), dan geeft `advies` je tijdens je beurt een tip:
welke kaart de computer op niveau "moeilijk" zou leggen en waarom, bij een boer welke kleur,
en een herinnering om "Laatste kaart!" te melden. De tip-kaart licht op in je hand.

In [ ]:
# ==============================================================
# COMPUTERSPELERS
# ==============================================================
def beste_kleur(hand: list[str]) -> str:
    """De kleur waarvan je de meeste kaarten hebt (boeren en jokers tellen niet mee)."""
    tellen = Counter(kleur_van(k) for k in hand if not is_joker(k) and waarde_van(k) != "B")
    if not tellen:
        return random.choice(list(KLEUREN))
    return max(KLEUREN, key=lambda kleur: tellen[kleur])   # gelijk: de eerste in KLEUREN


def waardeer(spel: dict, kaart: str, hand: list[str], niveau: str, toeval: bool = True) -> float:
    """Hoe graag een computer deze kaart legt (hoger = liever). Zonder toeval
    geeft dezelfde situatie altijd dezelfde keuze (voor de assistent)."""
    regels = spel["regels"]
    soort = pestsoort(kaart, regels)
    rest = [k for k in hand if k != kaart]
    aantal = len(spel["spelers"])
    volgende_speler = spel["spelers"][(spel["beurt"] + spel["richting"]) % aantal]

    score = 10.0
    if not is_joker(kaart):
        # Kaarten van een kleur waar je er veel van hebt, raak je makkelijk kwijt.
        score += sum(1 for k in rest if not is_joker(k) and kleur_van(k) == kleur_van(kaart))
    if soort in ("joker", "boer"):
        score = 2   # bewaren: die passen altijd
    elif soort == "twee":
        score = 6
    elif soort == "zeven":
        # Alleen fijn als je daarna nog iets kunt leggen.
        vervolg = [k for k in rest if not is_joker(k) and
                   (kleur_van(k) == kleur_van(kaart) or waarde_van(k) == "7")]
        score = 14 if vervolg else 4
    elif soort in ("acht", "aas"):
        score = 8
    if spel["te_pakken"] and soort == "twee":
        score = 12  # stapelen met een 2 en de joker bewaren

    if niveau == "moeilijk" and len(volgende_speler["hand"]) <= 2 and soort in ("twee", "joker", "acht"):
        score = 30  # de volgende is bijna uit: pesten!
    return score + (random.random() if toeval else 0)   # bij gelijke score: toeval


def computer_zet(spel: dict) -> None:
    """De computer die aan de beurt is, doet één zet: een kaart leggen, of
    pakken (en de gepakte kaart meteen spelen als dat kan)."""
    idx = spel["beurt"]
    speler = spel["spelers"][idx]
    if speler["soort"] != "computer":
        raise SpelFout("Er is geen computer aan de beurt.")
    niveau = speler["niveau"]

    speelbaar = speelbare_kaarten(spel)
    if not speelbaar:
        pak_kaart(spel, idx)
        if spel["beurt"] != idx or not spel["getrokken"]:
            return   # gepakt en gepast
        speelbaar = [spel["getrokken"]]

    if niveau == "makkelijk":
        kaart = random.choice(speelbaar)
    else:
        kaart = max(speelbaar, key=lambda k: waardeer(spel, k, speler["hand"], niveau))
    rest = [k for k in speler["hand"] if k != kaart]
    kleur = beste_kleur(rest) if pestsoort(kaart, spel["regels"]) == "boer" else None
    gemeld = len(rest) == 1 and random.random() >= VERGEET_KANS[niveau]
    speel_kaart(spel, idx, kaart, kleur, gemeld)


# ==============================================================
# ASSISTENT: tips voor de speler
# ==============================================================
def advies(spel: dict, idx: int) -> dict | None:
    """Een tip voor speler idx als die aan de beurt is: welke kaart (en bij een boer
    welke kleur) de computer op niveau 'moeilijk' zou kiezen, en waarom."""
    if spel["fase"] != "spelen" or spel["beurt"] != idx:
        return None
    regels = spel["regels"]
    hand = spel["spelers"][idx]["hand"]
    speelbaar = speelbare_kaarten(spel)
    if not speelbaar:
        if spel["te_pakken"]:
            tekst = f"Je hebt geen 2 of joker om bij te leggen: pak de {spel['te_pakken']} kaarten."
        elif regels["niet_uitgaan_pest"] and len(hand) == 1 and pestsoort(hand[0], regels):
            tekst = "Op een pestkaart mag je niet uitgaan: pak een kaart."
        else:
            tekst = "Geen enkele kaart past: pak een kaart van de stapel."
        return {"kaart": None, "kleur": None, "tekst": tekst}

    kaart = max(speelbaar, key=lambda k: (waardeer(spel, k, hand, "moeilijk", toeval=False), k))
    soort = pestsoort(kaart, regels)
    rest = [k for k in hand if k != kaart]
    aantal = len(spel["spelers"])
    volgende_speler = spel["spelers"][(idx + spel["richting"]) % aantal]
    naam = kaart_naam(kaart)
    kleur = beste_kleur(rest) if soort == "boer" else None

    if spel["te_pakken"]:
        extra = TWEE_PAKKEN if soort == "twee" else JOKER_PAKKEN
        tekst = (f"Leg {naam} erop: dan hoef jij er geen {spel['te_pakken']} te pakken, "
                 f"maar pakt {volgende_speler['naam']} er {spel['te_pakken'] + extra}.")
    elif soort in ("twee", "joker", "acht") and len(volgende_speler["hand"]) <= 2:
        tekst = (f"{volgende_speler['naam']} heeft nog maar {kaarten_tekst(len(volgende_speler['hand']))}: "
                 f"pest met {naam}!")
    elif soort == "zeven":
        tekst = f"Leg {naam}: daarna ben je nog een keer aan de beurt en je hebt nog een kaart die past."
    elif soort == "boer":
        tekst = (f"Leg {naam} en kies {SYMBOLEN[kleur]} {KLEUREN[kleur]}: "
                 f"daar heb je de meeste kaarten van.")
    elif soort == "joker":
        tekst = f"Alleen je joker past. Leg hem: {volgende_speler['naam']} pakt er {JOKER_PAKKEN}."
    elif len(speelbaar) == 1:
        tekst = f"Alleen {naam} past: leg die."
    else:
        zelfde = sum(1 for k in rest if not is_joker(k) and kleur_van(k) == kleur_van(kaart))
        tekst = f"Leg {naam}"
        if zelfde:
            tekst += f": je hebt nog {zelfde} andere {KLEUREN[kleur_van(kaart)]}, die raak je zo makkelijk kwijt"
        if any(pestsoort(k, regels) for k in speelbaar if k != kaart):
            tekst += ". Bewaar je pestkaarten voor later"
        tekst += "."
    if spel["getrokken"]:
        tekst = f"De kaart die je pakte past. {tekst}"
    if len(hand) == 2 and regels["laatste_kaart"]:
        tekst += " Vink eerst \"Laatste kaart!\" aan, anders pak je 2 strafkaarten."
    return {"kaart": kaart, "kleur": kleur, "tekst": tekst}

In [ ]:
# %%ALLEEN-NOTEBOOK%% — testcel: laat alleen computers spelen (de mens wordt ook een computer)
def speel_automatisch(instellingen: dict, toon: bool = False) -> dict:
    spel = nieuw_spel("Test", instellingen)
    spel["spelers"][0].update(soort="computer", niveau=instellingen["niveau"])
    zetten = 0
    while spel["fase"] == "spelen":
        computer_zet(spel)
        zetten += 1
        if zetten > 2000:
            raise RuntimeError("Het spel stopt niet")
    if toon:
        for regel in spel["log"]:
            print(regel["tekst"])
    totaal = sum(len(s["hand"]) for s in spel["spelers"]) + len(spel["trekstapel"]) + len(spel["aflegstapel"])
    assert totaal == 52 + (2 if instellingen["regels"]["joker"] else 0), "er zijn kaarten zoek"
    return spel


spel = speel_automatisch(STANDAARD_INSTELLINGEN, toon=True)

# 300 spellen met willekeurige instellingen: stopt het altijd en blijven alle kaarten in het spel?
winst = Counter()
for _ in range(300):
    instellingen = {**STANDAARD_INSTELLINGEN,
                    "computers": random.randint(MIN_COMPUTERS, MAX_COMPUTERS),
                    "niveau": random.choice(list(NIVEAUS)),
                    "startkaarten": random.randint(MIN_STARTKAARTEN, 7),
                    "regels": {r: random.random() < 0.7 for r in REGELS}}
    winst[speel_automatisch(instellingen)["winnaar"]] += 1
print("\n300 spellen gespeeld. Winnaars per plaats aan tafel:", dict(sorted(winst.items())))

# Moeilijk tegen makkelijk: speler 0 moeilijk, de rest makkelijk.
gewonnen = 0
for _ in range(300):
    spel = nieuw_spel("Test", {**STANDAARD_INSTELLINGEN, "computers": 2, "niveau": "makkelijk"})
    spel["spelers"][0].update(soort="computer", niveau="moeilijk")
    while spel["fase"] == "spelen":
        computer_zet(spel)
    gewonnen += spel["winnaar"] == 0
print(f"Moeilijk wint {gewonnen / 3:.0f}% tegen twee makkelijke spelers (eerlijk zou 33% zijn).")

# Online: drie mensen in de wachtkamer, twee computers erbij, en iemand gaat weg tijdens het spel.
spel = nieuw_spel("Anna", {**STANDAARD_INSTELLINGEN, "spelvorm": "online", "computers": 2}, "anna@test.nl")
assert spel["fase"] == "wachten"
meedoen(spel, "Bas", "bas@test.nl")
meedoen(spel, "anna", "anna2@test.nl")                  # dezelfde naam krijgt een nummer
assert meedoen(spel, "Bas", "bas@test.nl") == 1          # opnieuw meedoen: dezelfde plaats
try:
    begin_spel(spel, 1)
    raise AssertionError("alleen de maker mag beginnen")
except SpelFout as fout:
    print("Goed geweigerd:", fout)
begin_spel(spel, 0)
print("Aan tafel:", [s["naam"] + (" (computer)" if s["soort"] == "computer" else "") for s in spel["spelers"]])
assert verlaat(spel, 1) is False and spel["spelers"][1]["soort"] == "computer"
while spel["fase"] == "spelen":                          # iedereen laten spelen als computer
    for s in spel["spelers"]:
        s.setdefault("niveau", "normaal")
    aan_de_beurt(spel)["soort"] = "computer"
    computer_zet(spel)
print("Online test klaar, winnaar:", spel["spelers"][spel["winnaar"]]["naam"])

In [ ]:
# %%ALLEEN-NOTEBOOK%% — testcel: de assistent
spel = nieuw_spel("Jij", {**STANDAARD_INSTELLINGEN, "niveau": "makkelijk", "computers": 2})
while spel["beurt"] != 0:
    computer_zet(spel)
print("Jouw hand:", ", ".join(kaart_naam(k) for k in sorteer_hand(spel["spelers"][0]["hand"])))
print("Open kaart:", kaart_naam(bovenste(spel)))
print("Tip:", advies(spel, 0)["tekst"])

# Speel 300 keer als mens die altijd de tip volgt, tegen twee makkelijke computers.
gewonnen = 0
for _ in range(300):
    spel = nieuw_spel("Jij", {**STANDAARD_INSTELLINGEN, "niveau": "makkelijk", "computers": 2})
    while spel["fase"] == "spelen":
        if spel["beurt"] != 0:
            computer_zet(spel)
            continue
        tip = advies(spel, 0)
        if tip["kaart"]:
            speel_kaart(spel, 0, tip["kaart"], tip["kleur"], laatste_gemeld=len(spel["spelers"][0]["hand"]) == 2)
        elif spel["getrokken"]:
            pas(spel, 0)
        else:
            pak_kaart(spel, 0)
    gewonnen += spel["winnaar"] == 0
print(f"\nMet de tips van de assistent win je {gewonnen / 3:.0f}% (eerlijk zou 33% zijn).")

## 7 — Weergave
`spel_weergave` maakt van het spel wat één speler mag zien: zijn eigen hand (met welke
kaarten speelbaar zijn), van de anderen alleen het aantal kaarten, de stapels, de gevraagde
kleur en het logboek. Pas aan het einde van een ronde zie je de handen van de anderen.
`wachtkamer_weergave` is wat je ziet zolang een online spel nog niet begonnen is.

In [ ]:
# ==============================================================
# WEERGAVE: wat de pagina van het spel te zien krijgt
# ==============================================================
def kaart_weergave(kaart: str) -> dict:
    """Alles om een kaart te tekenen (zie de macro 'kaart' in spel.html)."""
    if is_joker(kaart):
        return {"code": kaart, "joker": True, "naam": "joker", "rood": False}
    kleur, waarde = kleur_van(kaart), waarde_van(kaart)
    return {"code": kaart, "joker": False, "naam": kaart_naam(kaart), "waarde": waarde,
            "symbool": SYMBOLEN[kleur], "rood": kleur in RODE_KLEUREN,
            "plaatje": waarde in WAARDE_NAMEN and waarde != "A"}


def status_tekst(spel: dict, idx: int) -> str:
    if spel["fase"] == "klaar":
        winnaar = spel["spelers"][spel["winnaar"]]["naam"]
        return "Jij hebt gewonnen!" if spel["winnaar"] == idx else f"{winnaar} heeft gewonnen."
    if spel["beurt"] != idx:
        return f"{aan_de_beurt(spel)['naam']} is aan de beurt…"
    if spel["getrokken"]:
        return f"Je pakte {kaart_naam(spel['getrokken'])}. Speel hem, of pas."
    if spel["te_pakken"]:
        if spel["regels"]["stapelen"]:
            return f"Leg een 2 of joker bij, of pak {spel['te_pakken']} kaarten."
        return f"Pak {spel['te_pakken']} kaarten."
    if spel["extra_beurt"]:
        return "Je bent nog een keer aan de beurt."
    return "Jij bent aan de beurt: leg een kaart of pak er een."


def spel_weergave(spel: dict, idx: int) -> dict:
    """Wat speler idx mag zien. Kaarten van anderen gaan hier niet in (alleen
    het aantal), zodat je ze niet in de broncode van de pagina kunt vinden."""
    aantal = len(spel["spelers"])
    mijn_beurt = spel["fase"] == "spelen" and spel["beurt"] == idx
    speelbaar = set(speelbare_kaarten(spel)) if mijn_beurt else set()
    ik = spel["spelers"][idx]
    tip = advies(spel, idx) if ik.get("assistent") else None

    hand = []
    for kaart in sorteer_hand(ik["hand"]):
        weergave = kaart_weergave(kaart)
        weergave.update(speelbaar=kaart in speelbaar, nieuw=kaart in ik["nieuw"],
                        tip=bool(tip) and kaart == tip["kaart"],
                        boer=pestsoort(kaart, spel["regels"]) == "boer")
        hand.append(weergave)

    # Tegenstanders in de volgorde waarin ze na jou aan tafel zitten.
    tegenstanders = []
    for stap in range(1, aantal):
        i = (idx + stap) % aantal
        speler = spel["spelers"][i]
        tegenstanders.append({
            "naam": speler["naam"],
            "aantal": len(speler["hand"]),
            "beurt": spel["fase"] == "spelen" and spel["beurt"] == i,
            "computer": spel["online"] and speler["soort"] == "computer",
            "gewonnen": spel["stand"][i],
            "winnaar": spel["winnaar"] == i,
            "hand": [kaart_weergave(k) for k in sorteer_hand(speler["hand"])]
                    if spel["fase"] == "klaar" else [],
        })

    stand = sorted(({"naam": s["naam"], "gewonnen": spel["stand"][i], "kaarten": len(s["hand"]),
                     "ik": i == idx} for i, s in enumerate(spel["spelers"])),
                   key=lambda r: (-r["gewonnen"], r["kaarten"]))
    return {
        "hand": hand,
        "tegenstanders": tegenstanders,
        "ik": {"naam": ik["naam"], "gewonnen": spel["stand"][idx], "beurt": mijn_beurt},
        "open": [kaart_weergave(k) for k in spel["aflegstapel"][-3:]],
        "kleur": spel["kleur"],
        "kleur_anders": (spel["kleur"] is not None and not is_joker(bovenste(spel))
                         and spel["kleur"] != kleur_van(bovenste(spel))),
        "te_pakken": spel["te_pakken"],
        "trekstapel": len(spel["trekstapel"]),
        "richting": spel["richting"],
        "status": status_tekst(spel, idx),
        "kan_pakken": mijn_beurt and not spel["getrokken"],
        "kan_passen": mijn_beurt and bool(spel["getrokken"]),
        "advies": tip,
        "laatste_kaart_regel": spel["regels"]["laatste_kaart"],
        "computer_aan_zet": spel["fase"] == "spelen" and aan_de_beurt(spel)["soort"] == "computer",
        "log": list(reversed(spel["log"][-8:])),
        "klaar": spel["fase"] == "klaar",
        "gewonnen": spel["winnaar"] == idx,
        "winnaar": spel["spelers"][spel["winnaar"]]["naam"] if spel["winnaar"] is not None else None,
        "stand": stand,
        "ronde": spel["ronde"],
        "online": spel["online"],
    }


def wachtkamer_weergave(spel: dict, idx: int) -> dict:
    """Wat speler idx in de wachtkamer ziet: wie er al meedoen en hoe er gespeeld wordt."""
    erbij = computers_erbij(spel)
    return {
        "spelers": [{"naam": s["naam"], "ik": i == idx, "maker": i == 0}
                    for i, s in enumerate(spel["spelers"])],
        "computers": erbij,
        "computer_namen": computer_namen(spel, erbij),
        "plaatsen_over": MAX_SPELERS - len(spel["spelers"]) - erbij,
        "maker": spel["spelers"][0]["naam"],
        "ik_maker": idx == 0,
        "kan_beginnen": len(spel["spelers"]) + erbij >= 2,
        "niveau": spel["niveau"],
        "startkaarten": spel["startkaarten"],
        "regels": [tekst for regel, tekst in REGELS.items() if spel["regels"][regel]],
    }

## 8 — De website (Flask)
De schermen:
1. **Aanmelden en instellingen** (`aanmelden.html`): e-mailadres, naam, spelvorm,
   tegenstanders. Achter het tandwiel (⚙) staan de startkaarten, de spelregels en de
   assistent. Zodra je een bekend e-mailadres invult, haalt de pagina je naam,
   instellingen en scores op (`/speler`). **Start spel** slaat alles op en deelt de kaarten.
2. **De wachtkamer** (`wachtkamer.html`, alleen online). Online is de id van het spel een korte
   **spelcode** (`CODE_TEKENS`, `CODE_LENGTE`); met `/?code=...` kom je op scherm 1 met de code ingevuld.
3. **Het spel** (`spel.html`). Achter het tandwiel (⚙) zet je de assistent aan of uit,
   ga je terug naar de instellingen of meld je je af.

Een zet is een formulier (`/speel`, `/pak`, `/pas`). Is een computer aan de beurt, dan
vraagt de pagina na `COMPUTER_PAUZE_MS` zelf om zijn zet (`/computer`); zo zie je elke zet
gebeuren. De spellen staan in het geheugen van de server (`SPELLEN`); de browser krijgt
alleen een spel-id in een cookie, dus niemand kan de kaarten van een ander zien.
Na een herstart van de server zijn lopende spellen weg (de database blijft).

**Online** zoekt elke browser zijn plaats aan tafel op met het e-mailadres in de sessie
(`mijn_plaats`). Elke verandering verhoogt de `versie` van het spel; de pagina vraagt elke
`VERVERS_MS` naar `/versie` en ververst als die anders is. Een computer doet online pas een zet
als de vorige zet lang genoeg geleden is, ook al vragen alle apparaten erom.

**Beeld en geluid** gaan rechtstreeks tussen de apparaten (WebRTC, `static/beeld.js`). De server
brengt alleen de verbindingsvoorstellen over (`/beeld/...`). Om elkaar via internet te vinden,
gebruiken de browsers een openbare STUN-server van Google (alleen de adressen, geen beeld).
In sommige netwerken (strenge bedrijfsfirewalls) lukt een rechtstreekse verbinding niet.

**Toegang**: staat er iets in `TOEGESTANE_EMAILS`, dan mogen alleen die adressen (of domeinen)
spelen. Andere adressen krijgen een melding, en ook geen naam of instellingen te zien. Haal je
iemand van de lijst, dan is die bij de volgende klik afgemeld. Er is geen wachtwoord: wie een
toegestaan adres kent, kan het gebruiken.

In [ ]:
# ==============================================================
# FLASK-APP
# ==============================================================
app = Flask(__name__,
            template_folder=str(SPEL_MAP / "templates"),
            static_folder=str(SPEL_MAP / "static"))
app.secret_key = os.getenv("FLASK_SECRET") or secrets.token_hex(32)
app.config["SESSION_COOKIE_SECURE"] = SPEL_HTTPS   # cookie alleen over https
app.config["SESSION_COOKIE_SAMESITE"] = "Lax"
if SPEL_HTTPS:
    # Achter Caddy: het IP-adres van de bezoeker staat in X-Forwarded-For (voor het auditlog).
    app.wsgi_app = ProxyFix(app.wsgi_app, x_for=1, x_proto=1, x_host=1)

# Alleen tegen de computer heeft een spel een lange willekeurige id; online is de
# id de korte spelcode die je aan de anderen geeft.
SPELLEN: dict[str, dict] = {}
SPELLEN_SLOT = threading.Lock()
SPEL_MAX_LEEFTIJD = 24 * 3600   # spellen waarin een dag niets gebeurde, worden opgeruimd


def mijn_plaats(spel: dict) -> int | None:
    """Waar jij aan tafel zit: de mens met jouw e-mailadres. Zo vindt elk apparaat
    zijn plaats terug, ook als er in de wachtkamer iemand weggaat."""
    for i, speler in enumerate(spel["spelers"]):
        if speler["soort"] == "mens" and speler["email"] == session.get("email"):
            return i
    return None


def huidig_spel() -> dict | None:
    """Het spel van deze browser, als je daar (nog) aan tafel zit."""
    spel = SPELLEN.get(session.get("spel_id", ""))
    return spel if spel and mijn_plaats(spel) is not None else None


def gewijzigd(spel: dict) -> None:
    """Na elke verandering. Aan de versie zien de andere apparaten dat ze moeten verversen."""
    spel["versie"] = spel.get("versie", 0) + 1
    spel["gewijzigd"] = time.time()


def nieuwe_code() -> str:
    while True:
        code = "".join(secrets.choice(CODE_TEKENS) for _ in range(CODE_LENGTE))
        if code not in SPELLEN:
            return code


def verlaat_huidig_spel() -> None:
    """Je gaat weg uit je spel. Tegen de computer verdwijnt het spel; online speelt
    een computer verder met je kaarten (of verdwijn je uit de wachtkamer)."""
    with SPELLEN_SLOT:
        spel_id = session.pop("spel_id", "")
        spel = SPELLEN.get(spel_id)
        idx = mijn_plaats(spel) if spel else None
        if idx is None:
            return
        spel.get("beeld", {}).pop(spel["spelers"][idx]["id"], None)
        if not spel["online"] or verlaat(spel, idx):
            del SPELLEN[spel_id]
        else:
            gewijzigd(spel)


# Pagina's die je zonder aanmelden mag openen.
OPEN_PAGINAS = {"index", "start", "speler_gegevens", "versie", "healthz", "static"}


GEEN_TOEGANG = "Dit e-mailadres heeft geen toegang tot dit spel. Vraag de beheerder om je toe te voegen."


def log(soort: str, email: str = "", details: str = "") -> None:
    """Auditlog met het IP-adres en de browser van dit verzoek."""
    audit(soort, email, request.remote_addr or "", details, request.headers.get("User-Agent", ""))


@app.before_request
def eerst_aanmelden():
    """Zonder e-mailadres in de sessie kom je eerst op het aanmeldscherm. Staat je
    adres niet (meer) op TOEGESTANE_EMAILS, dan word je afgemeld."""
    if "email" in session and not email_toegestaan(session["email"]):
        log("toegang_ingetrokken", session["email"])
        verlaat_huidig_spel()
        session.clear()
    if "email" not in session and request.endpoint not in OPEN_PAGINAS:
        return redirect(url_for("index"))


@app.context_processor
def speler_in_templates():
    return {"email": session.get("email"), "naam": session.get("naam"),
            "ververs_ms": VERVERS_MS, "beeld_ververs_ms": BEELD_VERVERS_MS}


def instellingen_uit_formulier() -> dict:
    vorm = request.form
    return {
        "spelvorm": vorm.get("spelvorm", "alleen"),
        "computers": vorm.get("computers", ""),
        "niveau": vorm.get("niveau", ""),
        "startkaarten": vorm.get("startkaarten", ""),
        "regels": {regel: True for regel in vorm.getlist("regel") if regel in REGELS},
        "assistent": vorm.get("assistent") == "1",
    }


def aanmeldscherm(email_tekst="", naam_tekst="", instellingen=None, speler=None, code_tekst=""):
    """Scherm 1: aanmelden en instellingen. Zonder instellingen: die van de
    aangemelde speler, of de standaard. Met een spelcode staat 'online' aan."""
    if instellingen is None:
        instellingen = speler["instellingen"] if speler else STANDAARD_INSTELLINGEN
    if code_tekst:
        instellingen = {**instellingen, "spelvorm": "online"}
    return render_template(
        "aanmelden.html", spel=None,
        email_tekst=email_tekst or (speler or {}).get("email", ""),
        naam_tekst=naam_tekst or (speler or {}).get("naam", ""),
        code_tekst=code_tekst, code_lengte=CODE_LENGTE,
        instellingen=instellingen, speler=speler, spelvormen=SPELVORMEN,
        niveaus=NIVEAUS, regels=REGELS, max_naam=MAX_NAAM, assistent_niveaus=ASSISTENT_NIVEAUS,
        min_computers=MIN_COMPUTERS, max_computers=MAX_COMPUTERS, max_spelers=MAX_SPELERS,
        min_startkaarten=MIN_STARTKAARTEN, max_startkaarten=MAX_STARTKAARTEN,
    )


@app.get("/")
def index():
    # Een link met ?code=... van een ander spel: eerst het aanmeldscherm om mee te doen.
    code = request.args.get("code", "").strip().upper()
    spel = huidig_spel()
    if spel is None or (code and code != session.get("spel_id")):
        speler = haal_speler(session["email"]) if "email" in session else None
        return aanmeldscherm(speler=speler, code_tekst=code)
    plaats = mijn_plaats(spel)
    if spel["fase"] == "wachten":
        return render_template("wachtkamer.html", spel=spel, pagina="wachtkamer",
                               w=wachtkamer_weergave(spel, plaats), niveaus=NIVEAUS,
                               max_computers=MAX_COMPUTERS)
    return render_template("spel.html", spel=spel, weergave=spel_weergave(spel, plaats),
                           symbolen=SYMBOLEN, kleuren=KLEUREN, pauze_ms=COMPUTER_PAUZE_MS,
                           assistent_mogelijk=assistent_mogelijk(spel),
                           assistent_aan=spel["spelers"][plaats].get("assistent"))


@app.post("/speler")
def speler_gegevens():
    """Aanmeldscherm: naam, instellingen en scores bij een e-mailadres, zodat
    ze al ingevuld zijn voordat je op Start klikt."""
    try:
        email = lees_email(request.form.get("email", ""))
    except ValueError:
        return {"bekend": False}
    if not email_toegestaan(email):
        log("opzoeken_geweigerd", email)
        return {"bekend": False, "geen_toegang": True}
    speler = haal_speler(email)
    if not speler:
        return {"bekend": False}
    return {"bekend": True, "naam": speler["naam"], "instellingen": speler["instellingen"],
            "gespeeld": speler["gespeeld"], "gewonnen": speler["gewonnen"]}


@app.post("/start")
def start():
    """Scherm 1 -> het spel (of online: de wachtkamer). Met een spelcode doe je mee
    met het spel van een ander; zonder code maak je zelf een spel."""
    email_tekst = request.form.get("email", "")
    naam_tekst = request.form.get("naam", "")
    code = request.form.get("code", "").strip().upper()
    ruwe_instellingen = instellingen_uit_formulier()
    try:
        try:
            email = lees_email(email_tekst)
        except ValueError:
            log("ongeldig_adres", email_tekst.strip())
            raise
        if not email_toegestaan(email):
            log("geweigerd", email)
            raise ValueError(GEEN_TOEGANG)
        naam = lees_naam(naam_tekst)
        if ruwe_instellingen["spelvorm"] == "online" and code:
            return meedoen_met_code(email, naam, code, ruwe_instellingen["assistent"])
        instellingen = controleer_instellingen(ruwe_instellingen)
        spel = nieuw_spel(naam, instellingen, email)
    except (ValueError, SpelFout) as fout:
        flash(str(fout), "fout")
        return aanmeldscherm(email_tekst, naam_tekst, ruwe_instellingen, code_tekst=code)
    # De keuze voor de assistent bewaren, ook als hij op dit niveau niet meedoet.
    bewaar_speler(email, naam, {**instellingen, "assistent": ruwe_instellingen["assistent"]})

    verlaat_huidig_spel()
    with SPELLEN_SLOT:
        grens = time.time() - SPEL_MAX_LEEFTIJD
        for oud in [k for k, s in SPELLEN.items() if s["gewijzigd"] < grens]:
            del SPELLEN[oud]
        spel_id = nieuwe_code() if spel["online"] else uuid.uuid4().hex
        spel["code"] = spel_id if spel["online"] else None
        gewijzigd(spel)
        SPELLEN[spel_id] = spel
    session.clear()
    session.update(email=email, naam=naam, spel_id=spel_id)
    log("aangemeld", email, f"maakt spel {spel_id}" if spel["online"] else "tegen de computer")
    return redirect(url_for("index"))


def meedoen_met_code(email: str, naam: str, code: str, assistent: bool):
    """Online meedoen met het spel van een ander (in de wachtkamer)."""
    with SPELLEN_SLOT:
        spel = SPELLEN.get(code)
        if spel is None or not spel["online"]:
            spel = None
    if spel is None:
        log("onbekende_code", email, f"code {code[:20]}")
        raise ValueError(f"Er is geen spel met de code {code[:20]}. Controleer de code.")
    with SPELLEN_SLOT:
        meedoen(spel, naam, email, assistent)
        gewijzigd(spel)
    speler = haal_speler(email)
    instellingen = speler["instellingen"] if speler else STANDAARD_INSTELLINGEN
    bewaar_speler(email, naam, {**instellingen, "spelvorm": "online", "assistent": assistent})
    if session.get("spel_id") != code:
        verlaat_huidig_spel()
    session.clear()
    session.update(email=email, naam=naam, spel_id=code)
    log("aangemeld", email, f"doet mee met spel {code}")
    return redirect(url_for("index"))


@app.post("/afmelden")
def afmelden():
    if "email" in session:
        log("afgemeld", session["email"])
    verlaat_huidig_spel()
    session.clear()
    return redirect(url_for("index"))


@app.post("/stoppen")
def stoppen():
    """Terug naar de instellingen (je blijft aangemeld)."""
    verlaat_huidig_spel()
    return redirect(url_for("index"))


def zet(handeling):
    """Voert een zet uit op het spel van deze browser en telt een afgelopen ronde
    voor alle mensen aan tafel in de database. Een zet die niet mag, wordt een
    melding. Geeft de handeling False terug, dan veranderde er niets."""
    spel = huidig_spel()
    if spel is None:
        return redirect(url_for("index"))
    try:
        with SPELLEN_SLOT:
            if handeling(spel) is not False:
                gewijzigd(spel)
            tellen = spel["fase"] == "klaar" and not spel.get("geteld")
            mensen = []
            if tellen:
                spel["geteld"] = True
                mensen = [(s["email"], i) for i, s in enumerate(spel["spelers"])
                          if s["soort"] == "mens" and s["email"]]
    except SpelFout as fout:
        flash(str(fout), "fout")
        return redirect(url_for("index"))
    for email, i in mensen:
        tel_ronde(email, gewonnen=spel["winnaar"] == i)
    return redirect(url_for("index"))


@app.post("/speel")
def speel():
    kaart = request.form.get("kaart", "")
    kleur = request.form.get("kleur") or None
    gemeld = request.form.get("laatste") == "1"
    return zet(lambda spel: speel_kaart(spel, mijn_plaats(spel), kaart, kleur, gemeld))


@app.post("/pak")
def pak_route():
    return zet(lambda spel: pak_kaart(spel, mijn_plaats(spel)))


@app.post("/pas")
def pas_route():
    return zet(lambda spel: pas(spel, mijn_plaats(spel)))


@app.post("/computer")
def computer():
    """Eén zet van de computer die aan de beurt is. De pagina vraagt hier zelf om
    (na COMPUTER_PAUZE_MS), zodat je elke zet ziet gebeuren. Online vragen alle
    apparaten erom; wie te snel na de vorige zet komt, wordt overgeslagen."""
    def handeling(spel):
        rust = time.time() - spel["gewijzigd"]
        if (spel["fase"] != "spelen" or aan_de_beurt(spel)["soort"] != "computer"
                or rust < COMPUTER_PAUZE_MS / 1000 * 0.7):
            return False
        computer_zet(spel)
    return zet(handeling)


@app.post("/begin")
def begin():
    """Wachtkamer: wie het spel maakte, deelt de kaarten."""
    return zet(lambda spel: begin_spel(spel, mijn_plaats(spel)))


@app.post("/computers")
def computers_kiezen():
    """Wachtkamer: het aantal computers dat meespeelt."""
    try:
        aantal = int(request.form.get("computers", ""))
    except ValueError:
        aantal = -1
    antwoord = zet(lambda spel: kies_computers(spel, mijn_plaats(spel), aantal))
    # Ook bewaren bij je instellingen, voor het volgende spel dat je maakt.
    speler = haal_speler(session["email"])
    if speler and 0 <= aantal <= MAX_COMPUTERS:
        bewaar_speler(session["email"], speler["naam"], {**speler["instellingen"], "computers": aantal})
    return antwoord


@app.get("/versie")
def versie():
    """Online: de pagina vraagt elke VERVERS_MS of er iets veranderd is."""
    spel = huidig_spel()
    return {"versie": spel["versie"] if spel else None}


@app.post("/assistent")
def assistent():
    """Tandwiel in het spel: je eigen assistent aan- of uitzetten (ook voor de volgende keer)."""
    spel = huidig_spel()
    if spel is None:
        return redirect(url_for("index"))
    if not assistent_mogelijk(spel):
        flash("Tegen de computer is de assistent er alleen op niveau makkelijk.", "fout")
        return redirect(url_for("index"))
    with SPELLEN_SLOT:
        ik = spel["spelers"][mijn_plaats(spel)]
        ik["assistent"] = not ik.get("assistent")
    speler = haal_speler(session["email"])
    if speler:
        bewaar_speler(session["email"], speler["naam"],
                      {**speler["instellingen"], "assistent": ik["assistent"]})
    flash(f"De assistent staat {'aan' if ik['assistent'] else 'uit'}.", "info")
    return redirect(url_for("index"))


@app.post("/volgende-ronde")
def volgende_ronde():
    def handeling(spel):
        if spel["fase"] != "klaar":
            raise SpelFout("De ronde is nog niet afgelopen.")
        spel["geteld"] = False
        nieuwe_ronde(spel)
    return zet(handeling)


# ==============================================================
# BEELD EN GELUID (online)
# ==============================================================
# Het beeld en geluid gaan rechtstreeks van apparaat naar apparaat (WebRTC,
# zie static/beeld.js). De server brengt alleen de korte berichten over waarmee
# twee apparaten een verbinding afspreken. spel["beeld"] = {id: {naam, post, gezien}}
# van iedereen die zijn camera aan heeft.
MAX_BEELD_BERICHT = 20_000   # tekens; een verbindingsvoorstel is een paar duizend


def beeld_deelnemers(spel: dict) -> dict:
    """Wie de camera aan heeft. Wie BEELD_WEG_NA seconden niets ophaalde, is weg."""
    beeld = spel.setdefault("beeld", {})
    grens = time.time() - BEELD_WEG_NA
    for weg in [i for i, d in beeld.items() if d["gezien"] < grens]:
        del beeld[weg]
    return beeld


def beeld_handeling(werk):
    """Voert werk(spel, ik, deelnemers) uit voor een mens in een online spel."""
    spel = huidig_spel()
    if spel is None or not spel["online"]:
        return {"fout": "Beeld en geluid zijn er alleen in een online spel."}, 400
    with SPELLEN_SLOT:
        ik = spel["spelers"][mijn_plaats(spel)]
        return werk(spel, ik, beeld_deelnemers(spel))


@app.post("/beeld/aan")
def beeld_aan():
    def werk(spel, ik, deelnemers):
        deelnemers[ik["id"]] = {"naam": ik["naam"], "post": [], "gezien": time.time()}
        return {"ik": ik["id"]}
    return beeld_handeling(werk)


@app.post("/beeld/uit")
def beeld_uit():
    def werk(spel, ik, deelnemers):
        deelnemers.pop(ik["id"], None)
        return {"ok": True}
    return beeld_handeling(werk)


@app.post("/beeld/stuur")
def beeld_stuur():
    """Een bericht (verbindingsvoorstel of antwoord) voor één andere deelnemer."""
    aan = request.form.get("aan", "")
    bericht = request.form.get("bericht", "")
    if len(bericht) > MAX_BEELD_BERICHT:
        return {"fout": "Bericht te lang."}, 400
    try:
        bericht = json.loads(bericht)
    except json.JSONDecodeError:
        return {"fout": "Geen geldig bericht."}, 400

    def werk(spel, ik, deelnemers):
        if ik["id"] in deelnemers and aan in deelnemers:
            deelnemers[aan]["post"] = deelnemers[aan]["post"][-20:] + [{**bericht, "van": ik["id"]}]
        return {"ok": True}
    return beeld_handeling(werk)


@app.post("/beeld/post")
def beeld_post():
    """Je berichten ophalen, en wie er nog meer in het gesprek zit."""
    def werk(spel, ik, deelnemers):
        mij = deelnemers.get(ik["id"])
        if mij is None:
            return {"ik": None, "deelnemers": [], "berichten": []}
        mij["gezien"] = time.time()
        berichten, mij["post"] = mij["post"], []
        return {"ik": ik["id"], "berichten": berichten,
                "deelnemers": [{"id": i, "naam": d["naam"]} for i, d in deelnemers.items()]}
    return beeld_handeling(werk)


@app.get("/healthz")
def healthz():
    return "ok"

## 9 — Server starten
Blokkeert de kernel. Open daarna http://127.0.0.1:5000. Onderbreek de cel (■) om te stoppen.

In [ ]:
# ==============================================================
# STARTEN
# ==============================================================
# In Docker komt `docker stop` binnen als SIGTERM. Python als hoofdproces
# (PID 1) negeert dat standaard, waardoor Docker 10 s wacht en dan hard
# stopt. Met deze handler stopt de server meteen. In Jupyter niet doen:
# daar regelt de kernel het stoppen zelf.
if "ipykernel" not in sys.modules:
    signal.signal(signal.SIGTERM, lambda *_: sys.exit(0))

if __name__ == "__main__":
    print(f"[Server] luistert op http://{SERVER_HOST}:{SERVER_PORT}")
    app.run(host=SERVER_HOST, port=SERVER_PORT, debug=False,
            use_reloader=False, threaded=True)